# 1. 构造LoRAConfig

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
  r = 16,
  lora_alpha = 16,
  #target_modules = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
  target_modules = "all-linear",
  lora_dropout= 0,
  task_type = "CAUSAL_LM"
)

/Users/entingwu/miniconda3/envs/finetune_qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W1103 16:56:30.325000 24995 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# 2. 通过get_peft_model获取一个可训练的PeftModel

In [2]:
from peft import get_peft_model
from transformers import AutoModelForCausalLM

original_model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1481.42it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1204.29it/s, Materializing param=model.norm.weight]                             
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
peft_model = get_peft_model(model, lora_config)

# 2.1 通过peft_model看到当前所需要训练的参数量是多少

In [4]:
peft_model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 761,724,928 || trainable%: 1.3250


# 2.2 看下PEFTModel和原模型的结构差异

# 2.2.1 原模型的结构

In [5]:
original_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [6]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=1024, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=1024, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
      

In [7]:
peft_model.base_model.model.model.layers[0].self_attn.q_proj.base_layer.weight.requires_grad

False

In [24]:
peft_model.base_model.model.model.layers[0].self_attn.q_proj.lora_A["default"].weight.requires_grad

True

# 3. 构造SFTConfig
# 只需要对learning rate进行调整

In [8]:
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs/06_sft"

config = SFTConfig(
  per_device_train_batch_size=4,
  per_device_eval_batch_size=4,
  gradient_accumulation_steps=8, # 4*8=32 更新参数
  max_steps=300,
  logging_strategy="steps",
  logging_steps=10,
  report_to="tensorboard",
  learning_rate=3e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  eval_strategy="steps",
  eval_steps=50,
  metric_for_best_model="eval_loss",
  greater_is_better=False,
  load_best_model_at_end=True,
  save_strategy="steps",
  save_steps=50,
  save_total_limit=2,
  output_dir="./finetuned/05_sft_demo",
  bf16=True,
  gradient_checkpointing=False,
  activation_offloading=False,
  max_length= 650,
  assistant_only_loss=True,
  chat_template_path="./new_chat_template.jinja"
)

In [10]:
from datasets import load_dataset, DatasetDict
## 1.1 数据加载
data:DatasetDict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})
data["train"] = data["train"].select(range(10000))

## 1.2 数据处理
from typing import Dict,List
def convert_type(examples:Dict[str, List]):
    """
    讲数据，转换成 SFTTrainer所需要的Language Modeling类型，对话格式
    """
    conversation_list:List[List[Dict]] = examples["conversation"]

    all_data_messages_list = []

    for data in conversation_list:
        human_message = data[0]["human"]
        assistant_message = data[0]["assistant"]

        message_list = [
            {"role":"user","content":human_message},
            {"role":"assistant","content":assistant_message}
        ]

        all_data_messages_list.append(message_list)

    return {"messages":all_data_messages_list}


# batched=True，传递给convert_type的是一批数据，
mapped_data = data.map(convert_type,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])
print('@', mapped_data["train"], mapped_data["test"])


@ Dataset({
    features: ['messages'],
    num_rows: 10000
}) Dataset({
    features: ['messages'],
    num_rows: 500
})


In [11]:
from trl.trainer.sft_trainer import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")
trainer = SFTTrainer(
  model=peft_model,
  args=config,
  processing_class=tokenizer,
  train_dataset=mapped_data["train"],
  eval_dataset=mapped_data["test"],
)

trainer.train()
# 保存模型参数，和Tokenizer相关的配置，从而使得，后面，可以通过加载这个路径，得到model和tokenizer
trainer.save_model("./finetuned/05_sft_demo")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1496.41it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 176127.66 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/entingwu/miniconda3/envs/finetune_qwen/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__in

Step,Training Loss,Validation Loss


KeyboardInterrupt: 